# Nesso-1 on Euler — your first run

This notebook takes you through one complete Nesso-1 run, start to finish, on 24 protein–ligand
pairs: four well-studied drug targets against six drugs. It takes **about 15 minutes**, most of
it waiting for one job.

By the end you will have a table of predicted binding affinities, a control you can check them
against, and a feel for the four steps so you can do it with your own pairs.

### What Nesso-1 does

Given a **protein sequence** and a **ligand SMILES** it predicts a binding affinity. No
alignment, no structure, no template. That is the appeal: it is roughly twenty times faster than
Boltz-2, which is what makes screening a whole organism's proteins against a whole metabolome
worth considering at all.

It is an **affinity model, not a structure model**. It does not give you a usable pose, and its
own preprint says the coarse-graining costs it accuracy on structural tasks. If you want a
structure, use Boltz-2 or AlphaFold3.

### Before you start

- Open this on Euler, so everything runs on the cluster. Any Python 3 kernel will do — every
  call into Nesso-1 goes through the shared image, so nothing has to be installed into your
  kernel, and it needs no GPU of its own.
- **There is nothing to install and nothing to clone.** Nesso-1 lives in a single Apptainer
  image that the whole group reads, and this notebook calls it through the `nesso` wrapper one
  directory up. If you can read `/cluster/work/beltrao/lucla/hpc-tools/nesso/`, you are ready —
  Step 0 checks precisely that.
- You need an account that can submit GPU jobs. Nothing here uses more than one GPU.

### One thing worth knowing up front

Euler's compute nodes have **no internet access by default**. Step 1 fetches sequences from
UniProt and structures from PubChem, so it has to run somewhere with a network: a login node,
with the `eth_proxy` module loaded. That is why that one cell runs through
`bash -lc 'module load eth_proxy && ...'` instead of calling the command directly. Everything the
GPU job itself needs is already inside the image. Forget the proxy and you get a connection
timeout, not a clear error.


In [ ]:
import os, subprocess, time
from pathlib import Path

def find_tools() -> Path:
    """Locate nesso_run.py, submit_nesso.sh and the `nesso` wrapper.

    Opened in place (hpc-tools/nesso/examples/), the search upwards finds them. Working from a
    COPY of this notebook in your own directory -- which is what you want in an IDE, where the
    shared checkout is read-only -- set HPC_TOOLS_NESSO_DIR to the checkout's nesso/ directory instead.
    """
    if os.environ.get("HPC_TOOLS_NESSO_DIR"):
        d = Path(os.environ["HPC_TOOLS_NESSO_DIR"]).expanduser().resolve()
        if not (d / "nesso_run.py").is_file():
            raise SystemExit(f"HPC_TOOLS_NESSO_DIR={d} has no nesso_run.py in it")
        return d
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "nesso_run.py").is_file():
            return d
    raise SystemExit(
        "Cannot find nesso_run.py above " + str(Path.cwd()) + ".\n"
        "You are running a copy of this notebook from outside the checkout, which is fine --\n"
        "point it at the tooling:\n"
        "    os.environ['HPC_TOOLS_NESSO_DIR'] = '/cluster/project/beltrao/lucla/repos/hpc-tools/nesso'\n"
        "in a cell above this one, or export it before starting the kernel.")


TOOL = find_tools()

USER   = os.environ["USER"]
DRIVER = str(TOOL / "nesso_run.py")
SUBMIT = str(TOOL / "submit_nesso.sh")

# The wrapper around the shared Apptainer image. `NESSO python ...` is the image's own
# interpreter, which is why this notebook needs no Python environment of yours: the kernel
# running these cells only ever launches subprocesses.
NESSO = str(TOOL / "nesso")
PY    = f"{NESSO} python"

# Everything per-pair lives on scratch. Only the final table is worth keeping, and for real work
# DEST belongs in project storage -- see the last section.
PAIRS = Path(f"/cluster/scratch/{USER}/nesso/first_run_pairs.csv")
RUN   = Path(f"/cluster/scratch/{USER}/nesso/first_run")
LOGS  = Path(f"/cluster/scratch/{USER}/nesso/slurm")
DEST  = Path(f"/cluster/scratch/{USER}/nesso/first_run_results")
for d in (LOGS, DEST):
    d.mkdir(parents=True, exist_ok=True)

def capture(cmd, **kw):
    """Run a shell command, stop the notebook if it fails, and hand back its stdout."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if p.stderr.strip():
        print("--- stderr ---"); print(p.stderr, end="")
    if p.returncode != 0:
        print(p.stdout or "", end="")
        raise RuntimeError(f"exit {p.returncode}")
    return p.stdout

def run(cmd, **kw):
    """capture(), but print the output instead of returning it.

    Returning nothing matters in a notebook: a cell whose last line is run(...) would otherwise
    print the output once and then echo the returned string again as the cell's result.
    """
    print("$", cmd)
    print(capture(cmd, **kw), end="")

print("tool :", TOOL)
print("run  :", RUN)

## Step 0 — check you can reach the image

Nesso-1, the model checkpoint, the chemical component dictionary and **ESM-2 650M** are all
inside one 5.5 GB Apptainer image. ESM-2 is the protein language model that turns a sequence
into the numeric embedding Nesso-1 actually reads — it is why no alignment is needed: the
evolutionary information Boltz-2 gets from an MSA, Nesso-1 gets from a model trained on a very
large number of sequences.

Having it as one shared, read-only file rather than an install each is deliberate. A `pip
install` of Nesso-1 is **41,358 files**, and `/cluster/project` runs out of *file-count* quota
long before it runs out of space, so a share can be full in files while terabytes of space sit
unused. Read-only also means nobody's analysis shifts under them because a colleague upgraded
something.

`weights_revision` is the Hugging Face revision of the weights, fixed when the image was built
and passed to every prediction. **Pinning it is the practice worth copying.** Without a pin, a
re-run months later can quietly pick up re-uploaded weights and hand you numbers that no longer
match the ones you already analysed, with nothing in the output saying anything changed. Here it
is stronger than a pin: the weights are *in* the image, and `BUILD_INFO.txt` carries a sha256 for
every one of them.

In [ ]:
info = capture(f"{NESSO} info")
# The record ends with a sha256 line per weight blob; the header is what matters here.
print(info.split("weight blobs")[0].rstrip())
print("\n(... then a sha256 for every weight blob in the image)")

# Kept for step 4, where the scores table records which weights produced it.
REVISION = next(l.split(":", 1)[1].strip()
                for l in info.splitlines() if l.startswith("weights_revision"))


## Step 1 — build a pairs table

The input is a **CSV with one row per protein–ligand pair**. Three columns are required:

| column | meaning |
|---|---|
| `pair_id` | unique, and usable as a filename: letters, digits, `.`, `_`, `-` |
| `protein_sequence` | one-letter amino acids |
| `ligand_smiles` | the ligand whose affinity is predicted |

Every **other** column is carried straight through to the output untouched, so your own
identifiers travel with the scores and you never have to join on row order afterwards. (There is
one optional column, `extra_ligands`, for putting further molecules into the complex — a metal
ion beside the macrocycle it sits in, say. See [`../README.md`](../README.md#the-contract).)

`example_pairs.py` builds such a table from public records: four drug targets fetched from
**UniProt by accession**, six drugs fetched from **PubChem by name**. Nothing is pasted into the
script, so swapping those two lists for your own makes your table. That is worth insisting on in
your own scripts too — a pasted sequence is one that can silently stop matching the accession it
claims to be, and nothing will tell you when it does.

This is the cell that needs the network.


In [ ]:
run(f"bash -lc 'module load eth_proxy >/dev/null 2>&1; "
    f"{PY} {TOOL}/example_pairs.py --out {PAIRS}'")

Four targets × six drugs = 24 pairs. Each drug is the established inhibitor of *one* of the four
targets, so every target gets at least one molecule that should bind and at least four with no
reason to. The `cognate` column says which is which.

That is a **positive and negative control built into the input**, and it is the part people skip.
A screen without one gives you a column of numbers and no way to tell signal from arithmetic: if
a score ranked the molecules by nothing but molecular weight, you would not be able to see it.
With the control in place, a working score has to put each target's own drug near the top, and
you can check that in one line.


In [ ]:
import pandas as pd
pairs = pd.read_csv(PAIRS)
print(pairs.shape, "- one row per pair\n")

view = pairs.copy()
view["protein_sequence"] = view["protein_sequence"].str.len().map(lambda n: f"<{n} aa>")
view["ligand_smiles"] = view["ligand_smiles"].str.slice(0, 28) + "..."
view.head(6)

## Step 2 — plan the run

`plan` reads the table and writes one small YAML per pair into a run directory **on scratch**,
grouped into **shards**. One shard = one array task = one GPU.

Two things happen here that are the whole reason this wrapper exists.

**Pairs are sharded by protein.** The plan sorts by sequence so one protein's pairs stay
together. Nesso-1 computes an ESM-2 embedding per protein, and keeping a protein's pairs in one
shard computes it once instead of once per shard the protein is scattered across.

**Every per-pair file stays on scratch.** Nesso-1 writes about **five files per pair** — the
input YAML, a structure `.npz`, a record `.json`, and the prediction. An all-against-all screen
of ~300 proteins × ~1,000 ligands would be roughly 1.5 million files. Shared project storage
runs out of **file-count quota** long before it runs out of space: such a share can sit within
a few percent of its file limit with terabytes still free. A run that writes its intermediates
there does not just fill your own share
— it can stop everyone in the group from writing anything at all. Your personal scratch has its
own 1-million-file quota, and that is where this belongs.


In [ ]:
run(f"{PY} {DRIVER} plan --pairs {PAIRS} --run-dir {RUN} --pairs-per-shard 100 --override")

`plan/pairs.csv` holds **one row per input row, always** — including any pair that could not be
planned, with `nesso:status` saying why. Nothing is ever dropped silently. An output table that
is shorter than its input has lost something and will not tell you what, and that is the failure
this rule exists to prevent: you find out six months later, when a count does not add up.

`--override` is doing one more thing worth knowing about: it deletes the previous run's inputs,
work and **shard tables** along with the plan. It has to. A shard whose table already exists is
skipped by the submit script, so a stale table left in place would be finalized as this run's
result — and if the pair ids happen to match, nothing in the output would say so.

One pair shape gets refused right here: a complex holding **two copies of the same molecule**
(ferric dicitrate, say — two citrates around an iron). Nesso-1 1.0.0 writes one conformer per
distinct molecule but then asks for one per ligand *chain*, so the second copy fails on the GPU.
The planner catches it first and the row comes back `skipped: repeated_ligand_entity`, which
costs you nothing instead of a queue wait.


In [ ]:
plan = pd.read_csv(RUN / "plan" / "pairs.csv")
print(plan["nesso:status"].value_counts().to_string())
print()
print((RUN / "plan" / "shards.csv").read_text())

y = sorted((RUN / "inputs").rglob("*.yaml"))[0]
print(f"--- {y.relative_to(RUN)} " + "-" * 40)
print(y.read_text())

## Step 3 — predict

One array task per shard. The `#SBATCH` lines inside `submit_nesso.sh` are **starting points for
a pilot, not measured values**, so the cell below overrides them — asking for more memory or
walltime than you need takes it from everyone else on the cluster, and asking for too little
means the job dies partway.

What a measured run of this shape looked like: a 600-pair shard of proteins of a median 238
residues on one RTX 4090 ran at **2.5 s/pair** and peaked at **11.5 GB of
host memory on 2 cores**. Memory is the binding constraint here — not cores, and not GPU memory.
So `--mem-per-cpu=8192` on 2 cores is 14 GB, which is that peak plus a little.

Two things make this example slower *per pair* than that, and both will apply to your own first
run:

- **Protein length.** These four targets run from 260 to 1,210 residues, not 238, and runtime
  grows steeply with length.
- **Fixed cost.** Loading the model and embedding the proteins is paid once, and here it is
  spread over 24 pairs instead of 600.

Expect something in the 15–25 s/pair range, and note that it depends on which card SLURM gives
you — the same work runs noticeably faster on an RTX 4090 than on a 3090. **Size your own run
from your own pilot**, which is what step 6 is for.


In [ ]:
n_shards = len(list((RUN / "inputs").iterdir()))
out = capture(f"sbatch --parsable --array=0-{n_shards - 1} "
              f"--time=00:30:00 --cpus-per-task=2 --mem-per-cpu=8192 "
              f"--gpus=1 --gres=gpumem:20g "
              f"{SUBMIT} {RUN}")
job = out.strip().split(";")[0].split("_")[0]
print("prediction job:", job, f"({n_shards} shard(s))")

Each array task predicts its shard, then **collapses the shard into one CSV and one zip and
deletes the shard's scratch tree** — but only if every pair was scored and the archive verified
(member count matches, and `unzip -t` passes). Peak file count then scales with how many shards
run *at once*, not with how big the run is. That is what makes a 300,000-pair screen possible on
a 1-million-file quota.

A shard that **fails keeps every one of its files**, because a failure is exactly when you need
to look at them. And a task that finds its shard's CSV already there exits without recomputing,
so a half-finished array is resubmitted as-is rather than from the start.

The next cell blocks until the job leaves the queue. On a free GPU this is a couple of minutes;
with a queue it is however long the queue is.


In [ ]:
def wait_for(job, poll=20, label=""):
    """Block until a SLURM job (or array) has left the queue, printing its state as it goes."""
    seen = None
    while True:
        states = subprocess.run(["squeue", "-j", str(job), "-h", "-o", "%T"],
                                capture_output=True, text=True).stdout.split()
        if not states:
            break
        summary = ", ".join(f"{s}x{states.count(s)}" for s in sorted(set(states)))
        if summary != seen:
            print(f"  {label}{summary}", flush=True); seen = summary
        time.sleep(poll)
    print(f"  {label}left the queue")
    # sacct is the only place to see how a job ENDED; squeue forgets it the moment it finishes.
    print(subprocess.run(["sacct", "-j", str(job), "-X", "-n", "-P",
                          "--format=JobID,State,Elapsed,MaxRSS"],
                         capture_output=True, text=True).stdout, end="")

wait_for(job, poll=20, label="predict: ")

The job log says which GPU it landed on, which weights revision it used, and its own
seconds-per-pair. Record that last number: runtime varies several-fold between card types, so a
walltime that was ample on one GPU can be tight on another.


In [ ]:
import re
log = sorted(LOGS.glob(f"nesso-{job}_*.out"))[-1]
# The prediction draws a progress bar, so the raw log is full of terminal escape codes.
text = re.sub(r"\x1b\[[0-9;?]*[a-zA-Z]", "", log.read_text())
lines = [ln for ln in text.splitlines() if ln.strip()]
bar = [i for i, ln in enumerate(lines) if ln.lstrip().startswith("Predicting ")]
lines = [ln for i, ln in enumerate(lines) if i not in set(bar[:-1])]   # keep only the final bar
print(log.name, "\n")
print("\n".join(lines))

## Step 4 — finalize

`finalize` collects the shard tables into one scores table, merges the per-shard archives into
one, writes a manifest so a pair can be found without opening the zip, and records a provenance
file: which pairs table (with its md5), which code commit, which model revision, which seed,
which GPU.

It refuses to run while any shard is unfinished, so you cannot accidentally publish a half-run
and then analyse it believing it is whole.


In [ ]:
run(f"{PY} {DRIVER} finalize --run-dir {RUN} --dest {DEST} --label first_run "
    f"--model-revision {REVISION}")

## Step 5 — read the scores

One row per input pair, with your own columns still attached. The ones to read:

| column | meaning |
|---|---|
| `nesso:affinity_pred_value` | log<sub>10</sub>(IC<sub>50</sub> in µM). **Lower is stronger**: −3 ≈ 1 nM, 0 ≈ 1 µM, +2 ≈ 100 µM |
| `nesso:affinity_probability_binary` | P(binder), 0–1. **Higher** means more likely a binder |
| `nesso:entropy_crop_pl` | how sharply the model placed the ligand at the interface. The model card calls this the confidence to read, and **0.0 means it could not place it** — do not trust that pair's score |
| `nesso:status` | `scored`, or why there is no score |

Note that `affinity_pred_value` and `affinity_probability_binary` are two separate heads of the
model, not two views of one number. They can disagree, and when they do, neither is obviously
right.

One caveat to attach to every number you read here: **Nesso-1 does not reproduce its own scores
exactly.** Running these same 24 pairs again, with the same seed and the same weights, moves
`affinity_pred_value` by up to 0.25 for a few pairs. It seeds its random choices but never asks
for deterministic CUDA kernels, so the arithmetic itself varies. So compare pairs *within* one
run, and never extend a finished set with a later job and rank the combined table —
[`../README.md`](../README.md#scores-are-not-reproducible-between-runs) has the numbers.


In [ ]:
scores = pd.read_csv(DEST / "first_run_nesso_scores.csv")
print(scores["nesso:status"].value_counts().to_string(), "\n")

# Lower = stronger predicted binding. Rows are targets, columns are drugs.
affinity = scores.pivot(index="target", columns="ligand_name",
                        values="nesso:affinity_pred_value")
affinity.round(2)

Now the control. For each target, is its own drug the strongest-looking of the six?

The check below is deliberately conservative in one respect: it takes the **worst** rank among
tied scores, so a tie counts as a miss rather than a lucky win. Tie-handling sounds like a
detail, but a score that returns the same value for many pairs will look good under optimistic
ranking and the difference can be the whole result.

Chance level, if the scores were random: with 6 ligands and 1 or 2 cognate ones per target, the
expected number of targets whose top-ranked ligand is cognate is 4 × (mean of 2/6, 2/6, 1/6,
1/6) = **1.0 of 4**.


In [ ]:
hits = []
for target, grp in scores.groupby("target"):
    grp = grp.sort_values("nesso:affinity_pred_value")          # lower = stronger
    best = grp.iloc[0]
    # Worst-case rank of the best cognate ligand: ties count against us.
    cog = grp[grp["cognate"] == 1]
    rank = int((grp["nesso:affinity_pred_value"]
                <= cog["nesso:affinity_pred_value"].min()).sum())
    hits.append({"target": target,
                 "top ligand": best["ligand_name"],
                 "top score": round(best["nesso:affinity_pred_value"], 2),
                 "best cognate rank (of 6)": rank,
                 "hit@1": bool(best["cognate"])})
hits = pd.DataFrame(hits)
print(hits.to_string(index=False))
print(f"\nhit@1: {hits['hit@1'].sum()} of {len(hits)}   (chance = 1.0)")

### What this does and does not show

If all four came out right, that tells you **your install is sound and the model works on the
chemistry it knows**. It does *not* tell you the model generalises. These are famous
drug–target pairs that are almost certainly in its training data, so the test is in-domain by
construction. Treating an in-domain result as evidence of generalisation is one of the easiest
mistakes to make with a pretrained model, and one of the hardest to spot afterwards.

Change the chemistry and the answer can change completely. Small, polar, charged ligands are
nothing like a kinase inhibitor, and an affinity model's ranking can collapse to chance on a
class it saw little of in training while a different model does fine on the identical pairs.

So: use the example to check your install, and then run your own controls on your own chemistry
before you believe anything.


## Step 6 — what it cost, and what is left on disk

`seff` reads the job's real usage. Size your next run from this, not from the numbers in the
submit script. Raise a request only after you have *seen* a job hit the limit — an out-of-memory
kill, or a walltime timeout that was not caused by something hanging.

One caveat that trips people up: SLURM records **host** memory (`MaxRSS`), never GPU memory. The
`--gres=gpumem:` request can only be confirmed by jobs *not* failing on it, so raise that one
only after an actual CUDA out-of-memory error.


In [ ]:
run(f"seff {job}_0 2>/dev/null || sacct -j {job} --format=JobID,State,Elapsed,MaxRSS,ReqMem,ReqCPUS")

And the point of the whole design: the per-pair files are gone, and what remains is a handful of
files you would be happy to keep forever.


In [ ]:
left = sum(1 for f in RUN.rglob("*") if f.is_file())
print(f"files still under {RUN}: {left}   (the per-pair trees are collapsed and deleted)\n")
for p in sorted(DEST.iterdir()):
    print(f"  {p.stat().st_size / 1e6:8.2f} MB  {p.name}")

The raw model output is in the zip rather than loose, for the file-count reason above. A zip's
index means you can still pull one pair out without unpacking the rest — look its member up in
the manifest, then:

```bash
unzip -p first_run_nesso_raw.zip <member>
```


In [ ]:
import zipfile
with zipfile.ZipFile(DEST / "first_run_nesso_raw.zip") as zf:
    members = zf.namelist()
    print(f"{len(members)} members; first few:")
    for m in members[:5]:
        print("  ", m)
    print("\n--- one affinity record " + "-" * 30)
    one = next(m for m in members if m.endswith("affinity.json"))
    print(one)
    print(zf.read(one).decode())

---

## Doing this with your own pairs

Replace step 1 with your own table and nothing else changes:

1. Write a CSV with `pair_id`, `protein_sequence`, `ligand_smiles`, plus any columns of your own.
   Build it with a script that fetches from the source, the way `example_pairs.py` does.
2. `plan --pairs <yours> --run-dir /cluster/scratch/$USER/nesso/<label> --pairs-per-shard 2000`
3. `sbatch --array=0` — **one shard as a pilot** — then `seff` it, then submit the rest with the
   sizes you measured.
4. `finalize --dest <project dir> --label <label>`
5. Archive the SLURM logs next to the results, so they do not sit loose on scratch:

   ```bash
   hpc-tools/archive_run_logs.sh archive-dir /cluster/scratch/$USER/nesso/slurm \
       <project dir> <label> --non-fatal
   ```

A few things to carry across:

- **`--pairs-per-shard` sets your parallelism and your peak file count.** 2,000 pairs per shard
  with 8 shards running at once is ~80,000 files on scratch at peak, against a 1-million quota.
- **Re-run the whole thing, never part of it.** Scores from two jobs are not comparable to
  better than about 0.25, so a set topped up with a later job cannot be ranked as one table.
- **Do not mix kernel settings inside one run.** `NESSO_KERNELS=1` turns on the cuEquivariance
  kernels and is the obvious lever if throughput matters, but it changes the numerics. Re-run the
  whole thing rather than part of it, or you are comparing scores that were not produced the same
  way.
- **Keep `DEST` in project storage**, not scratch — scratch is purged periodically, and the
  scores table plus its provenance is the thing you will still want next year.

### Tidying up

This example left its run directory and its pairs table on scratch. Both are finished with:

```bash
rm -rf /cluster/scratch/$USER/nesso/first_run /cluster/scratch/$USER/nesso/first_run_pairs.csv
```

The installed environment and weights cache are worth keeping — they are what makes the next run
start immediately.


In [ ]:
print(f"pairs table   : {PAIRS}")
print(f"run directory : {RUN}        <- safe to delete")
print(f"results       : {DEST}        <- move to project storage for real work")
print(f"logs          : {LOGS}")